# Práctica 6 — WGCNA: redes de coexpresión en el modelo 5xFAD de Alzheimer

**Módulo de Transcriptómica — Máster en Bioinformática**

En esta práctica usamos **PyWGCNA** para construir una red de coexpresión génica sobre un RNA-seq del modelo murino de Alzheimer **5xFAD**:

- **192 muestras** de corteza e hipocampo
- **2 genotipos**: `5xFADWT` (control) y `5xFADHEMI` (modelo de Alzheimer)
- **4 edades**: 4, 8, 12 y 18 meses
- Ambos sexos

**Flujo de la práctica:**

1. Preprocesado de la matriz de expresión
2. Construcción de la red y detección de módulos (`findModules`)
3. Relación módulo–rasgo (*module-trait relationships*)
4. Genes hub de cada módulo
5. Visualización de redes de coexpresión
6. Enriquecimiento funcional de los módulos (GO/KEGG)
7. Interacciones proteína–proteína (STRING)

> **Concepto clave**: a diferencia del análisis de expresión diferencial (gen a gen), WGCNA agrupa genes en **módulos** de genes coexpresados y relaciona cada módulo (resumido por su *eigengene*) con los rasgos experimentales.

## 1. Instalación

Instalamos PyWGCNA y sus dependencias. Tarda 1–2 minutos.

> Si Colab muestra un aviso de que hay que **reiniciar la sesión** tras la instalación, hazlo (*Entorno de ejecución → Reiniciar sesión*) y continúa desde la celda siguiente (no hace falta reinstalar).

In [ ]:
!pip install -q PyWGCNA
import importlib.metadata
print("PyWGCNA", importlib.metadata.version("PyWGCNA"))

## 2. Datos

Los datos son los del tutorial oficial de PyWGCNA (artículo del modelo 5xFAD), incluidos en el repositorio del curso:

- `expressionList.csv.gz`: matriz de expresión (192 muestras × ~22.000 genes, valores tipo TPM)
- `sampleInfo.csv`: metadatos de las muestras (`Age`, `Tissue`, `Sex`, `Genotype`)
- `geneList`: tabla de equivalencia Ensembl → símbolo de gen (para no depender del servicio BioMart, que a veces falla)

In [ ]:
import urllib.request
import gzip
import shutil
from pathlib import Path

DATA_DIR = Path("datos_5xFAD")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/victorsanchezarevalo/transcriptomica-master/main/practicas/06-wgcna-5xfad/datos"
FICHEROS = ["expressionList.csv.gz", "sampleInfo.csv", "geneList"]

for f in FICHEROS:
    destino = DATA_DIR / f
    if not destino.exists():
        print(f"Descargando {f} ...")
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", destino)
    print(f"{f}: {destino.stat().st_size / 1e6:.2f} MB")

# Descomprimir la matriz de expresión
gz = DATA_DIR / "expressionList.csv.gz"
csv = DATA_DIR / "expressionList.csv"
if not csv.exists():
    print("Descomprimiendo expressionList.csv.gz ...")
    with gzip.open(gz, "rb") as f_in, open(csv, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
print(f"expressionList.csv: {csv.stat().st_size / 1e6:.1f} MB")
print("\nDatos listos.")

## 3. Crear el objeto PyWGCNA

El objeto `WGCNA` guarda la expresión en formato **AnnData**: filas/`obs` = muestras, columnas/`var` = genes.

In [ ]:
import pandas as pd
import numpy as np
import PyWGCNA

pyWGCNA_5xFAD = PyWGCNA.WGCNA(
    name="5xFAD",
    species="mus musculus",
    geneExpPath=str(DATA_DIR / "expressionList.csv"),
    outputPath="PyWGCNA_output",
    save=True,
)

print(pyWGCNA_5xFAD.geneExpr)
pyWGCNA_5xFAD.geneExpr.to_df().iloc[:5, :5]

## 4. Preprocesado

`preprocess()` elimina:

- genes con demasiados valores faltantes o **baja expresión** (por defecto, < 1 TPM en la mayoría de muestras)
- muestras con demasiados valores faltantes
- genes con **varianza cero**

Estos filtros se pueden ajustar con `TPMcutoff` y `cut`.

In [ ]:
pyWGCNA_5xFAD.preprocess()
print("\nTras el preprocesado:")
print(pyWGCNA_5xFAD.geneExpr)

## 5. Selección de genes para la red

La matriz tiene **~55.000 genes**. Construir la red con todos requeriría una matriz de adyacencia de 55.000 × 55.000 (~25 GB de RAM) — inviable en Colab.

**Práctica estándar en WGCNA**: quedarse con los genes **más variables** entre muestras (los invariables no aportan información de coexpresión). Seleccionamos los **5.000 genes con mayor varianza**, lo que reduce la memoria a ~200 MB.

> En R/WGCNA se usa el mismo criterio (típicamente top 5000–10000 por MAD o varianza).

In [ ]:
# Selección de los 5000 genes más variables
var_genes = pyWGCNA_5xFAD.geneExpr.to_df().var(axis=0).sort_values(ascending=False)
TOP_N = 5000
top_genes = var_genes.head(TOP_N).index
print(f"Genes seleccionados: {len(top_genes)} de {pyWGCNA_5xFAD.geneExpr.shape[1]}")

pyWGCNA_5xFAD.geneExpr = pyWGCNA_5xFAD.geneExpr[:, top_genes].copy()
print(pyWGCNA_5xFAD.geneExpr)

## 6. Construcción de la red y detección de módulos

`findModules()` ejecuta en una sola llamada todo el flujo clásico de WGCNA:

1. **Elección del soft-thresholding power (β)**: se escoge la potencia más baja que da una topología aproximadamente *scale-free*
2. Similitud de coexpresión y matriz de **adyacencia**
3. **TOM** (Topological Overlap Matrix)
4. Clustering jerárquico sobre la disimilitud TOM + corte dinámico del dendrograma
5. **Fusión** de módulos muy similares

Encontraréis los parámetros usados en `pyWGCNA_5xFAD.sft` (powers explorados) y los módulos asignados en `geneExpr.var['moduleColors']`.

> Esta celda puede tardar **varios minutos**. Es la parte pesada del análisis.

In [ ]:
pyWGCNA_5xFAD.findModules()

# Módulos detectados y tamaño
modulos = pyWGCNA_5xFAD.geneExpr.var["moduleColors"].value_counts()
print(f"\n{len(modulos)} módulos detectados (incluido 'grey', los genes no asignados):")
print(modulos)

## 7. Añadir metadatos de muestras y de genes

Asignamos colores a los metadatos para las figuras, y añadimos el **símbolo de gen** a partir de la tabla local `geneList` (evita la consulta a BioMart).

In [ ]:
# Metadatos de las muestras
pyWGCNA_5xFAD.updateSampleInfo(path=str(DATA_DIR / "sampleInfo.csv"), sep=",")

# Colores para las figuras
pyWGCNA_5xFAD.setMetadataColor("Sex", {"Female": "green", "Male": "gold"})
pyWGCNA_5xFAD.setMetadataColor("Genotype", {"5xFADWT": "darkviolet", "5xFADHEMI": "deeppink"})
pyWGCNA_5xFAD.setMetadataColor("Age", {"4mon": "lightblue", "8mon": "deepskyblue",
                                       "12mon": "steelblue", "18mon": "navy"})
pyWGCNA_5xFAD.setMetadataColor("Tissue", {"Cortex": "salmon", "Hippocampus": "seagreen"})

pyWGCNA_5xFAD.geneExpr.obs.head()
# Edad como variable continua + combinación edad x sexo
pyWGCNA_5xFAD.geneExpr.obs["Timepoint"] = (
    pyWGCNA_5xFAD.geneExpr.obs["Age"]
    .replace({"4mon": 4, "8mon": 8, "12mon": 12, "18mon": 18})
    .astype(int)
)
pyWGCNA_5xFAD.geneExpr.obs["Age_Sex"] = (
    pyWGCNA_5xFAD.geneExpr.obs["Age"] + "_" + pyWGCNA_5xFAD.geneExpr.obs["Sex"]
)



In [ ]:
# Información de genes: Ensembl ID -> símbolo (tabla local)
geneList = pd.read_csv(DATA_DIR / "geneList", sep=r"\s+")
print(geneList.head())

pyWGCNA_5xFAD.updateGeneInfo(geneList)
pyWGCNA_5xFAD.geneExpr.var.head()

## 8. Relación módulo–rasgo

`analyseWGCNA()` calcula, para cada módulo:

1. Su **eigengene** (primer componente principal del módulo)
2. La **correlación módulo–rasgo** con los metadatos (con p-valor)
3. La relación entre *module membership* y *gene significance*

El heatmap módulo × rasgo es la figura central de un WGCNA.

In [ ]:
pyWGCNA_5xFAD.analyseWGCNA()

### Explorando el heatmap módulo–rasgo

La función `module_trait_relationships_heatmap()` permite repetir el análisis con las variables que queramos.

**Truco**: si queremos ver la edad como una sola fila (correlación continua), la convertimos a numérica creando una columna `Timepoint`. También podemos crear variables combinadas, como `Age_Sex`.

In [ ]:
# Heatmap con todos los metadatos (cada categoría como columna dummy)
pyWGCNA_5xFAD.module_trait_relationships_heatmap(
    metaData=["Age", "Tissue", "Sex", "Genotype"],
    alternative="two-sided",
)

In [ ]:
pyWGCNA_5xFAD.module_trait_relationships_heatmap(
    metaData=["Timepoint", "Tissue", "Sex", "Genotype"],
    alternative="two-sided",
)

> **Interpretación**: buscad los módulos con mayor correlación (positiva o negativa) con `Genotype_5xFADHEMI` (efecto de la enfermedad) y con `Timepoint` (progresión con la edad). ¿Coinciden?

## 9. Genes hub

Los **genes hub** son los más conectados dentro de cada módulo — candidatos a reguladores clave. `top_n_hub_genes()` los ordena por conectividad.

Elegimos automáticamente el **módulo más grande** (distinto de `grey`), pero cambiad `modulo_interes` por el módulo que más os interese del heatmap anterior (p. ej. el más correlacionado con el genotipo).

In [ ]:
# Módulo de interés: el más grande (excluyendo 'grey', los no asignados)
todos = pyWGCNA_5xFAD.geneExpr.var["moduleColors"]
modulos_validos = todos[todos != "grey"].value_counts()
modulo_interes = modulos_validos.index[0]
print("Módulos por tamaño:")
print(modulos_validos)
print(f"\nMódulo de interés: {modulo_interes} ({modulos_validos.iloc[0]} genes)")
print("→ Cambiad 'modulo_interes' por el módulo que queráis explorar.")

In [ ]:
hub = pyWGCNA_5xFAD.top_n_hub_genes(moduleName=modulo_interes, n=20)
cols = [c for c in ["gene_name", "moduleColors", "kME", "connectivity"] if c in hub.columns]
hub[cols].head(20) if cols else hub.head(20)

## 10. Visualización de la red de coexpresión

`CoexpressionModulePlot()` dibuja la red de un módulo (o de varios, mostrando las conexiones entre ellos). Los nodos más grandes/centrales suelen ser los genes hub.

> Las figuras se guardan en `PyWGCNA_output/figures/` y se muestran en pantalla.

In [ ]:
# Red del módulo de interés (top genes y conexiones)
pyWGCNA_5xFAD.CoexpressionModulePlot(
    modules=[modulo_interes],
    numGenes=20,
    numConnections=100,
    minTOM=0,
)

# pyvis guarda la red como HTML interactivo; la mostramos dentro del notebook
from IPython.display import HTML
HTML(filename=f"PyWGCNA_output/figures/network/{modulo_interes}.html")

In [ ]:
# Red con varios módulos para ver sus interconexiones
top_modulos = modulos_validos.index[:4].tolist()
print("Módulos representados:", top_modulos)
pyWGCNA_5xFAD.CoexpressionModulePlot(
    modules=top_modulos,
    numGenes=40,
    numConnections=400,
    minTOM=0,
    fileName="top_modulos",
)
HTML(filename="PyWGCNA_output/figures/network/top_modulos.html")

## 11. Enriquecimiento funcional de los módulos

`functional_enrichment_analysis()` lanza un ORA (vía Enrichr/gseapy) sobre los genes de un módulo, igual que hicimos en la práctica 5. Usamos librerías de ratón:

- **GO**: `GO_Biological_Process_2021` (+ CC y MF)
- **KEGG**: `KEGG_2019_Mouse`
- **Reactome**

> Necesita conexión a internet (consulta Enrichr).

In [ ]:
pyWGCNA_5xFAD.figureType = "png"

# Gene Ontology
pyWGCNA_5xFAD.functional_enrichment_analysis(
    type="GO",
    moduleName=modulo_interes,
    sets=["GO_Biological_Process_2021",
          "GO_Cellular_Component_2021",
          "GO_Molecular_Function_2021"],
    p_value=0.05,
)

In [ ]:
# KEGG (ratón)
pyWGCNA_5xFAD.functional_enrichment_analysis(
    type="KEGG",
    moduleName=modulo_interes,
    sets=["KEGG_2019_Mouse"],
    p_value=0.05,
)

## 12. Interacciones proteína–proteína (STRING)

Con `request_PPI()` recuperamos de **STRING** las interacciones conocidas entre los genes de un módulo, y con `PPI_network()` las dibujamos como red. `species=10090` es el código NCBI de *Mus musculus*.

> Necesita conexión a internet (consulta STRING).

In [ ]:
# PPIs conocidas entre los genes del módulo de interés
# Usar los top hub genes del módulo para evitar errores de URL muy larga en STRING
hub_genes = pyWGCNA_5xFAD.top_n_hub_genes(moduleName=modulo_interes, n=50)
genes_modulo = hub_genes["gene_name"].dropna().tolist() if "gene_name" in hub_genes else hub_genes.index.tolist()
print(f"Genes a consultar en STRING: {len(genes_modulo)}")

PPI = pyWGCNA_5xFAD.request_PPI(genes=genes_modulo, species=10090)
print(f"Interacciones encontradas: {len(PPI)}")
PPI.head(10)

In [ ]:
# Red PPI con los genes más conectados
top_genes_ppi = PPI.gene1.value_counts().head(20).index.tolist()
pyWGCNA_5xFAD.PPI_network(species=10090, geneList=top_genes_ppi)

## 13. Guardar y descargar resultados

Guardamos el objeto PyWGCNA completo (se puede recargar con `PyWGCNA.readWGCNA("5xFAD.p")` sin repetir el análisis) y empaquetamos todas las figuras y tablas.

In [ ]:
import shutil
from google.colab import files

# Guardar el objeto completo
pyWGCNA_5xFAD.saveWGCNA()

# Empaquetar resultados (figuras + tablas + objeto)
zip_path = shutil.make_archive("resultados_WGCNA_5xFAD", "zip", "PyWGCNA_output")
print(f"ZIP creado: {zip_path}")
files.download(zip_path)

## 14. Preguntas de reflexión

1. **Soft power**: ¿qué β eligió `findModules()` y por qué? ¿Qué pasaría con un β demasiado bajo o demasiado alto?
2. **Módulo–rasgo**: ¿qué módulos se asocian al genotipo 5xFADHEMI? ¿Y a la edad (`Timepoint`)? ¿Son los mismos? ¿Qué implicaría biológicamente?
3. **Tejido y sexo**: ¿hay módulos específicos de tejido o de sexo? ¿Cómo lo comprobaríais con la variable combinada `Age_Sex`?
4. **Genes hub**: mirad los genes hub del módulo más asociado al genotipo. ¿Conocéis alguno relacionado con la enfermedad de Alzheimer?
5. **Enriquecimiento**: ¿qué procesos biológicos dominan el módulo de interés? ¿Encajan con la patología (inflamación, sinapsis, metabolismo amiloide…)?
6. **WGCNA vs expresión diferencial**: ¿qué ventajas y limitaciones tiene el análisis por módulos frente al análisis gen a gen de las prácticas anteriores?

---

**Entrega**: el notebook ejecutado (o script `.py`) + un PDF con las figuras clave y las respuestas a las preguntas.